In [ ]:
" primary functions implementation "
import signal
from datetime import timedelta
import torch
import torch.distributed as dist
from torch.multiprocessing import ProcessContext

mp_context:ProcessContext = None

def _find_available_port() -> bool:
    """find available port."""
    import socket
    port = 29500
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('localhost', port)) != 0:
                return port
            port += 1

def kill_all(sig, frame):
    "sig stands for signal.SIGNAL_RECEIVED, frame never used."
    if mp_context!=None:
        procs = mp_context.processes
        # every single p stands for multiprocessing.process.BaseProcess
        for p in procs:
            if p.is_alive():
                p.kill() # kill -9 <process>
                # p.terminate() # kill <process>
    signal.signal(sig, signal.SIG_DFL) # restore default handler
    signal.raise_signal(sig) # Send a signal to the executing process.

# set signal handler
# which will invoke kill_all once receive signal.SIGTERM from outside
signal.signal(signal.SIGTERM, kill_all)

def _broadcast_inputs(rank: int, inputs: Any, stream: torch.cuda.Stream):
    """get input tensor parallel."""
    # broadcast meta info
    if rank != 0:
        inputs = [None, None, None, None]

    with torch.cuda.stream(stream):
        dist.broadcast_object_list(inputs)
    return inputs

def cache_swapping(cache_engine: CacheEngine, swap_in_map: dict,
                   swap_out_map: dict):
    """perform cache swapping."""
    issued_cache_op = False
    if len(swap_in_map) > 0:
        cache_engine.swap_in(swap_in_map)
        issued_cache_op = True
    if len(swap_out_map) > 0:
        cache_engine.swap_out(swap_out_map)
        issued_cache_op = True

    if issued_cache_op:
        cache_events = cache_engine.events
        for event in cache_events:
            event.wait()

def _tp_model_loop(
    rank: int,
    model_path: str,
    model_config: ModelConfig,
    cache_config: CacheConfig,
    backend_config: BackendConfig,
    world_size: int,
):
    "Start model loops for tensor parallel model inference."
    stream = torch.cuda.Stream()
    patched_model, cache_engine, _ = _tp_build_model(rank,
                                                     model_path,
                                                     model_config,
                                                     cache_config,
                                                     backend_config,
                                                     adapters=None,
                                                     world_size=world_size)

    while True:
        inputs, swap_in_map, swap_out_map, exit_flag = _broadcast_inputs(
            rank, None, stream)

        if exit_flag:
            break

        cache_swapping(cache_engine,
                       swap_in_map=swap_in_map,
                       swap_out_map=swap_out_map)

        model_forward(
            patched_model,
            inputs,
            cache_engine,
            world_size=world_size,
            stream=stream,
        )

def _start_tp_process(proc_id:int,
                      world_size:int,
                    # device_context: DeviceContext,
                      *args,
                      **kwargs):
    rank = proc_id + 1
    dist.init_process_group(
        'nccl', rank=rank, world_size=world_size,
        timeout=timedelta(days=35600)
    )
    torch.cuda.set_device(rank)
    with torch.inference_mode():


In [ ]:
import torch
from torch import multiprocessing as mp
from torch.multiprocessing import ProcessContext

import os
mp_ctx = mp.get_context('spawn')
mp_context:ProcessContext = None

###########################
#### start_sub_process ####
###########################
tensor_parallel = 2
world_size = tensor_parallel

os.environ.setdefault('MASTER_ADDR','127.0.0.1')
os.environ.setdefault('MASTER_PORT',_find_available_port())
addr = os.environ['MASTER_ADDR']
port = os.environ['MASTER_PORT']


device_context = "cuda"
mp_context = mp.spawn(
    
)
